# Notebook 4  Forecasting Models

## Overview

Daily energy consumption (`avg_kwh`) is forecast for three municipalities using a progression of models from simple baselines up to deep learning. All models are evaluated on a held-out test set using MAE, RMSE, MAPE, and sMAPE. The split is 70 % train / 15 % validation / 15 % test (chronological, no shuffling).

---

## Models and Hyperparameters

### 1. Naive Forecast

Predicts the last observed value for every future step. No parameters. Serves as a lower-bound baseline.

---

### 2. Seasonal Naive

Predicts the value observed exactly `period` days ago (same calendar position).

| Parameter | Value | Meaning |
|---|---|---|
| `period` | 365 | Look back one year to capture annual seasonality |

---

### 3. Rolling Mean (7d)

Predicts the mean of the last 7 training observations for every test step. No tunable parameters beyond the fixed window of 7 days.

---

### 4. Ridge Regression

Linear model with L2 regularisation fit on the full 39-feature matrix (calendar, lag, rolling, holiday flags).

| Parameter | Value |
|---|---|
| `alpha` | 1.0 |
| Preprocessing | `StandardScaler` |

---

### 5. Random Forest

Ensemble of decision trees fit on the same 39-feature matrix.

| Parameter | Value |
|---|---|
| `n_estimators` | 300 |
| `max_depth` | 12 |
| `min_samples_leaf` | 3 |
| `random_state` | 42 |
| `n_jobs` | âˆ’1 (all cores) |

---

### 6. SARIMA

Seasonal ARIMA fitted only on the target series (no exogenous features). Grid-searched over all combinations of the orders below; best model selected by test RMSE.

| Hyperparameter | Values tried |
|---|---|
| Non-seasonal order `(p, d, q)` | `(1,1,1)`, `(2,1,1)`, `(1,1,2)` |
| Seasonal order `(P, D, Q, s)` | `(1,1,1,7)`, `(0,1,1,7)` |
| `enforce_stationarity` | False |
| `enforce_invertibility` | False |
| `maxiter` | 150 |

Series with `min < 0.5` are fit in `log1p`-space and back-transformed with `expm1` to stabilise differencing.

---

### 7. SARIMAX

SARIMA extended with exogenous regressors. Fixed order; exogenous features include calendar, lag, and rolling statistics.

| Parameter | Value |
|---|---|
| Non-seasonal order | (1, 1, 1) |
| Seasonal order | (1, 1, 1, 7) |
| Exogenous features | `is_weekend`, `day_of_week`, `month`, `quarter`, `week_of_year`, `is_holiday_es`, `is_bridge_day`, `lag_1d`, `lag_7d`, `roll7_mean`, `roll30_mean`, `wow_change`, `dod_change`, `zscore_municipality`, `pct_rank_global` |
| `maxiter` | 150 |

Same log-space transformation as SARIMA when `min < 0.5`.

---

### 8. LSTM v2 â€” Dual-Stream

Two parallel input streams feed a shared prediction head. The sequential stream (LSTM) sees dynamic lag/rolling features over a lookback window; the static stream (linear encoder) sees the calendar features of the prediction day, which are known in advance.

```
Lookback window (seq_len Ã— |HISTORY_FEATURES|)
      â””â”€â–º LSTM (n_layers, hidden)
               â””â”€â–º last hidden state â”€â”€â”
                                       â”œâ”€â–º Linear head â†’ Å·
Prediction-day (|CALENDAR_FEATURES|)  â”‚
      â””â”€â–º Linear + ReLU + LayerNorm â”€â”€â”˜
```

| Parameter | Value |
|---|---|
| `seq_length` | 30 |
| LSTM `hidden` | 48 |
| LSTM `n_layers` | 2 |
| `dropout` | 0.3 |
| Static encoder output | 32 |
| Head hidden | 64 |
| Optimiser | AdamW (`lr=1e-3`, `weight_decay=1e-4`) |
| `batch_size` | 64 |
| `epochs` | 80 |
| Early-stopping `patience` | 15 |
| Gradient clipping | norm â‰¤ 1.0 |

---

### 9. N-BEATS v2 â€” Covariate-Conditioned

Stack of residual blocks. Each block receives the current residual (target channel) concatenated with the last-step neural features as a conditioning vector, then predicts a backcast (to subtract from the residual) and a forecast contribution.

```
residual_t  (seq_len,)
covariates  (|NEURAL_FEATURES|,)  â† last time-step features
    â””â”€â–º concat â†’ FC tower â†’ backcast + forecast contribution
```

| Parameter | Value |
|---|---|
| `seq_length` | 30 |
| `n_blocks` | 3 |
| `hidden_size` | 64 |
| FC layers per block | 4 |
| `dropout` | 0.2 |
| Optimiser | AdamW (`lr=1e-3`, `weight_decay=1e-4`) |
| `batch_size` | 64 |
| `epochs` | 80 |
| Early-stopping `patience` | 15 |

---

### 10. Hybrid v2 â€” Seasonal Naive + ResidualLSTM

Decomposes forecasting into a deterministic seasonal component and a learned residual correction:

`Å·(t) = SeasonalNaive(t, period=365) + ResidualLSTM(window of past residuals + features)`

The ResidualLSTM sees a rolling window where each step is `[normalised_residual_t, calendar_features_t, lag_features_t]`, giving the corrector temporal context over the error signal.

| Parameter | Value |
|---|---|
| Seasonal component period | 365 days |
| ResidualLSTM `hidden` | 32 |
| ResidualLSTM `n_layers` | 1 |
| `dropout` | 0.3 |
| `seq_length` | 30 |
| Optimiser | AdamW (`lr=1e-3`, `weight_decay=1e-4`) |
| `batch_size` | 64 |
| `epochs` | 80 |
| Early-stopping `patience` | 15 |

# Energy Forecasting â€” Baselines, SARIMA/SARIMAX, LSTM, N-BEATS and Hybrid Residual NN
## v2 â€” Feature-Aware Neural Models

**What changed in v2:**
- Cyclical sin/cos encodings for day-of-week, month, week-of-year (smoother than raw integers)
- Explicit feature split: `CALENDAR_FEATURES` (known at forecast time) vs `HISTORY_FEATURES` (lagged)
- **LSTM v2** â€” Dual-stream: sequential LSTM on dynamic features + static encoder for day-ahead calendar features
- **N-BEATS v2** â€” Covariates (calendar + lags) injected into every block as a conditioning vector
- **Hybrid v2** â€” `ResidualLSTM`: sees a window of `[residual_t, calendar_t, lag_features_t]` instead of a tabular snapshot

## v3 â€” Bug Fixes: Leakage, Overfit, SARIMA Log-Space

**What changed in v3:**
- **Data leakage fix**: `log_kwh` removed from `feature_cols` â€” it is `np.log1p(target_t)`, a same-day transform of the target; Ridge/RF had direct access to the answer
- **PATIENCE 100 â†’ 15**: the old value effectively disabled early stopping; all 80 epochs ran while val diverged; training now halts after 15 epochs without improvement
- **LSTM hidden 64â†’48, dropout 0.2â†’0.3**: smaller capacity + stronger regularisation to address Vitoria val/train â‰ˆ12Ã—
- **N-BEATS hidden 128â†’64, dropout added to each FC block**: reduces moderate overfit (val/train â‰ˆ4.6Ã—)
- **ResidualLSTM n_layers 2â†’1, hidden 64â†’32, dropout 0.2â†’0.3**: corrector was deeper than warranted for a small residual signal
- **SARIMA/SARIMAX log-space**: series with min < 0.05 (e.g. Vitoria-Gasteiz) are fit in log1p space then expm1-inverted; stabilises differencing and prevents MAPE/sMAPE explosion


In [17]:
# 0. Imports and config
import warnings, random, gc, math
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from statsmodels.tsa.statespace.sarimax import SARIMAX

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Turn off while testing if it is too heavy
RUN_SARIMA = True
RUN_SARIMAX = True
RUN_LSTM = True
RUN_NBEATS = True
RUN_HYBRID = True

EPOCHS = 80
BATCH_SIZE = 64
PATIENCE = 15
SEQ_LENGTH = 30
FORECAST_SEASONAL_PERIOD = 365  # diÃ¡rio: 7 semanal; muda para 365 para yearly seasonal naive
SARIMA_SEASONAL_PERIOD = 7     # 365 fica muito pesado em SARIMA/SARIMAX

Device: cpu


## 1. Load data

Altera `DATA_PATH` para o teu CSV. O notebook tenta detetar automaticamente:
- target: `kwh_per_user`, `avg_kwh`, `kWh`, `kwh`
- grupo: `municipality`, `_source_folder`, `user_id`, `id`


In [18]:
BASE_PATH = Path(r'C:/Users/GONCA/Desktop/Iscte/MCD/Theses')
DATA_PATH = BASE_PATH / 'results' / 'data' / 'municipality_daily_consumption.csv'

DATE_COL = 'date'
TARGET_CANDIDATES = ['avg_kwh']
GROUP_CANDIDATES = ['municipality', '_source_folder', 'user_id', 'id']

def pick_first_existing(columns, candidates, label):
    for c in candidates:
        if c in columns:
            return c
    raise ValueError(f'Could not find {label}. Tried: {candidates}')

df_raw = pd.read_csv(DATA_PATH)
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])
df_raw = df_raw.sort_values(DATE_COL).reset_index(drop=True)

# Compute per-user energy density if not already present
if 'avg_kwh_per_user' not in df_raw.columns:
    df_raw['avg_kwh_per_user'] = df_raw['avg_kwh'] / df_raw['n_users'].replace(0, float('nan'))

TARGET_COL = pick_first_existing(df_raw.columns, TARGET_CANDIDATES, 'target column')
GROUP_COL = pick_first_existing(df_raw.columns, GROUP_CANDIDATES, 'group column')

print('Shape:', df_raw.shape)
print('Date range:', df_raw[DATE_COL].min().date(), 'â†’', df_raw[DATE_COL].max().date())
print('Target:', TARGET_COL)
print('Group:', GROUP_COL)
display(df_raw.head())

Shape: (115144, 6)
Date range: 2014-11-04 â†’ 2022-06-08
Target: avg_kwh
Group: municipality


,municipality,date,avg_kwh,std_kwh,n_users,avg_kwh_per_user
0,Santander,2014-11-04,0.138348,0.155339,3,0.046116
1,Santander,2014-11-05,0.218616,0.202234,4,0.054654
2,Santander,2014-11-06,0.184656,0.145579,4,0.046164
3,Santander,2014-11-07,0.198615,0.154671,4,0.049654
4,Santander,2014-11-08,0.152052,0.135469,4,0.038013


## 2. Feature engineering

Se as colunas jÃ¡ existirem, sÃ£o mantidas. Se faltarem, sÃ£o criadas quando possÃ­vel.

In [19]:
FEATURE_COLUMNS_REQUESTED = [
    'is_weekend', 'day_of_week', 'month', 'quarter', 'week_of_year',
    'is_holiday_es', 'is_bridge_day',
    'lag_1d', 'lag_7d', 'lag_14d', 'lag_30d',
    'roll7_mean', 'roll7_std', 'roll30_mean', 'roll7_ratio',
    'wow_change', 'dod_change',
    'zscore_municipality', 'pct_rank_global', 'avg_kwh_per_user'  # log_kwh removed â€” np.log1p(target_t) is same-day leakage
]

def add_calendar_features(df):
    df = df.copy()
    df['day_of_week']  = df[DATE_COL].dt.dayofweek
    df['day_name']     = df[DATE_COL].dt.day_name()
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['month']        = df[DATE_COL].dt.month
    df['month_name']   = df[DATE_COL].dt.month_name()
    df['quarter']      = df[DATE_COL].dt.quarter
    df['week_of_year'] = df[DATE_COL].dt.isocalendar().week.astype(int)
    # â”€â”€ v2: cyclical encodings (smooth, no ordinal gap at boundary) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    df['sin_dow']   = np.sin(2 * np.pi * df['day_of_week']  / 7)
    df['cos_dow']   = np.cos(2 * np.pi * df['day_of_week']  / 7)
    df['sin_month'] = np.sin(2 * np.pi * df['month']        / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month']        / 12)
    df['sin_week']  = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['cos_week']  = np.cos(2 * np.pi * df['week_of_year'] / 52)
    return df

def add_es_holidays(df):
    df = df.copy()
    try:
        import holidays
        years = sorted(df[DATE_COL].dt.year.unique())
        es_holidays = holidays.country_holidays('ES', years=years)
        holiday_dates = set(pd.to_datetime(list(es_holidays.keys())))
        df['is_holiday_es'] = df[DATE_COL].isin(holiday_dates).astype(int)
        next_day = df[DATE_COL] + pd.Timedelta(days=1)
        prev_day = df[DATE_COL] - pd.Timedelta(days=1)
        df['is_bridge_day'] = (
            ((df[DATE_COL].dt.dayofweek == 0) & next_day.isin(holiday_dates)) |
            ((df[DATE_COL].dt.dayofweek == 4) & prev_day.isin(holiday_dates))
        ).astype(int)
    except Exception:
        df['is_holiday_es'] = df.get('is_holiday_es', 0)
        df['is_bridge_day'] = df.get('is_bridge_day', 0)
    return df

def add_lags_rollings(df, target_col, group_col):
    df = df.sort_values([group_col, DATE_COL]).copy()
    g  = df.groupby(group_col, group_keys=False)[target_col]
    for lag in [1, 7, 14, 30]:
        col = f'lag_{lag}d'
        if col not in df.columns:
            df[col] = g.shift(lag)
    if 'roll7_mean'  not in df.columns: df['roll7_mean']  = g.shift(1).rolling(7,  min_periods=1).mean().reset_index(level=0, drop=True)
    if 'roll7_std'   not in df.columns: df['roll7_std']   = g.shift(1).rolling(7,  min_periods=2).std().reset_index(level=0, drop=True)
    if 'roll30_mean' not in df.columns: df['roll30_mean'] = g.shift(1).rolling(30, min_periods=1).mean().reset_index(level=0, drop=True)
    if 'roll7_ratio' not in df.columns: df['roll7_ratio'] = df['lag_1d'] / df['roll7_mean'].replace(0, np.nan)
    if 'wow_change'  not in df.columns: df['wow_change']  = (df['lag_1d'] - df['lag_7d'])   / df['lag_7d'].replace(0, np.nan)
    if 'dod_change'  not in df.columns:
        lag_2d = g.shift(2)
        df['dod_change'] = (df['lag_1d'] - lag_2d) / lag_2d.replace(0, np.nan)
    if 'zscore_municipality' not in df.columns:
        exp_mean = g.expanding(min_periods=30).mean().shift(1).reset_index(level=0, drop=True)
        exp_std  = g.expanding(min_periods=30).std().shift(1).reset_index(level=0, drop=True)
        df['zscore_municipality'] = (df[target_col] - exp_mean) / exp_std.replace(0, np.nan)
    if 'pct_rank_global' not in df.columns:
        df['pct_rank_global'] = df.groupby(DATE_COL)[target_col].rank(pct=True)
    if 'log_kwh' not in df.columns:
        df['log_kwh'] = np.log1p(df[target_col].clip(lower=0))
    return df

df = add_calendar_features(df_raw)
df = add_es_holidays(df)
df = add_lags_rollings(df, TARGET_COL, GROUP_COL)
df = df.replace([np.inf, -np.inf], np.nan)

cat_cols = [c for c in ['day_name', 'month_name'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols, drop_first=False)

feature_cols = [c for c in FEATURE_COLUMNS_REQUESTED if c in df.columns and c != TARGET_COL]
feature_cols += [c for c in df.columns if c.startswith('day_name_') or c.startswith('month_name_')]
feature_cols = sorted(set(feature_cols))
# Remove avg_kwh_per_user from features when it IS the target (would be leakage)
if TARGET_COL == 'avg_kwh_per_user' and 'avg_kwh_per_user' in feature_cols:
    feature_cols.remove('avg_kwh_per_user')

print('Total feature_cols:', len(feature_cols))

Total feature_cols: 39


### Feature Sets for Neural Models

We separate features into three buckets based on **availability at inference time**
and **information type**:

| Set | Contents | Used in |
|---|---|---|
| `CALENDAR_FEATURES` | sin/cos encodings, holiday flags â€” **known in advance** | LSTM static stream, N-BEATS conditioning, Hybrid LSTM |
| `HISTORY_FEATURES` | lags, rolling stats â€” **already shifted**, safe | LSTM sequential stream, N-BEATS conditioning, Hybrid LSTM |
| `LEAKY_FEATURES` | z-score, pct_rank using same-day cross-section | **Excluded** from neural models |


In [20]:
# â”€â”€ v2: explicit feature-set definitions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Known for the prediction day (no leakage)
CALENDAR_FEATURES = [f for f in [
    'is_weekend', 'is_holiday_es', 'is_bridge_day',
    'sin_dow', 'cos_dow', 'sin_month', 'cos_month', 'sin_week', 'cos_week'
] if f in df.columns]

# Pre-lagged history (already shifted by â‰¥1 day â€” safe)
HISTORY_FEATURES = [f for f in [
    'lag_1d', 'lag_7d', 'lag_14d', 'lag_30d',
    'roll7_mean', 'roll7_std', 'roll30_mean', 'roll7_ratio',
    'wow_change', 'dod_change'
] if f in df.columns]

# Used in SARIMAX / Ridge / RF but excluded from neural models (cross-section leakage)
LEAKY_FEATURES = ['zscore_municipality', 'pct_rank_global']

# Safe neural feature set
NEURAL_FEATURES = CALENDAR_FEATURES + HISTORY_FEATURES

print(f'CALENDAR_FEATURES  ({len(CALENDAR_FEATURES)}): {CALENDAR_FEATURES}')
print(f'HISTORY_FEATURES   ({len(HISTORY_FEATURES)}):  {HISTORY_FEATURES}')
print(f'NEURAL_FEATURES    ({len(NEURAL_FEATURES)}):   total')


CALENDAR_FEATURES  (9): ['is_weekend', 'is_holiday_es', 'is_bridge_day', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month', 'sin_week', 'cos_week']
HISTORY_FEATURES   (10):  ['lag_1d', 'lag_7d', 'lag_14d', 'lag_30d', 'roll7_mean', 'roll7_std', 'roll30_mean', 'roll7_ratio', 'wow_change', 'dod_change']
NEURAL_FEATURES    (19):   total


## 3. Select groups and split

In [21]:
MIN_OBSERVATIONS = 365
N_GROUPS = 3

valid_groups = df.groupby(GROUP_COL).size().sort_values(ascending=False)
valid_groups = valid_groups[valid_groups >= MIN_OBSERVATIONS]
# GROUPS_TO_FORECAST = valid_groups.head(N_GROUPS).index.tolist()
# Manual example:
GROUPS_TO_FORECAST = ['Vitoria-Gasteiz', 'Donostia/San Sebastian', 'Pamplona/Iruna']

print('Groups selected:', GROUPS_TO_FORECAST)

def make_daily_group_frame(df, group_value):
    part = df[df[GROUP_COL] == group_value].sort_values(DATE_COL).copy().set_index(DATE_COL)
    full_idx = pd.date_range(part.index.min(), part.index.max(), freq='D')
    part = part.reindex(full_idx)
    part.index.name = DATE_COL
    part[GROUP_COL] = group_value
    part[TARGET_COL] = part[TARGET_COL].interpolate('linear').ffill().bfill()
    for c in feature_cols:
        if c not in part.columns:
            part[c] = np.nan
        part[c] = part[c].ffill().bfill()
        part[c] = part[c].fillna(part[c].median() if pd.api.types.is_numeric_dtype(part[c]) else 0)
    return part[[GROUP_COL, TARGET_COL] + feature_cols]

def time_split(part, train_ratio=0.70, val_ratio=0.15):
    n = len(part)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return part.iloc[:train_end].copy(), part.iloc[train_end:val_end].copy(), part.iloc[val_end:].copy()

series_data = {}
for group in GROUPS_TO_FORECAST:
    part = make_daily_group_frame(df, group)
    train, val, test = time_split(part)
    series_data[group] = {'full': part, 'train': train, 'val': val, 'test': test}
    print(f'\n{group}')
    print(f'  train: {train.index.min().date()} â†’ {train.index.max().date()} | {len(train)}')
    print(f'  val:   {val.index.min().date()} â†’ {val.index.max().date()} | {len(val)}')
    print(f'  test:  {test.index.min().date()} â†’ {test.index.max().date()} | {len(test)}')


Groups selected: ['Vitoria-Gasteiz', 'Donostia/San Sebastian', 'Pamplona/Iruna']

Vitoria-Gasteiz
  train: 2017-01-07 â†’ 2020-10-20 | 1383
  val:   2020-10-21 â†’ 2021-08-12 | 296
  test:  2021-08-13 â†’ 2022-06-05 | 297

Donostia/San Sebastian
  train: 2017-01-24 â†’ 2020-10-25 | 1371
  val:   2020-10-26 â†’ 2021-08-15 | 294
  test:  2021-08-16 â†’ 2022-06-05 | 294

Pamplona/Iruna
  train: 2017-05-29 â†’ 2020-12-01 | 1283
  val:   2020-12-02 â†’ 2021-09-02 | 275
  test:  2021-09-03 â†’ 2022-06-05 | 276


## 4. Metrics and helpers

In [22]:
all_results = {g: {} for g in GROUPS_TO_FORECAST}

def evaluate_model(actual, predicted, model_name):
    actual = pd.Series(actual).astype(float)
    predicted = pd.Series(predicted, index=actual.index).astype(float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.nanmean(np.abs((actual - predicted) / actual.replace(0, np.nan))) * 100
    smape = np.nanmean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted)).replace(0, np.nan)) * 100
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape, 'sMAPE(%)': smape}

def save_result(group, model_name, actual, pred, extra=None):
    metrics = evaluate_model(actual, pred, model_name)
    all_results[group][model_name] = {
        'pred': pd.Series(pred, index=actual.index),
        'metrics': metrics,
        'extra': extra or {}
    }
    return metrics

def seasonal_naive_forecast(history, horizon_index, period=7):
    history = pd.Series(history).copy()
    preds = []
    for date in horizon_index:
        candidate = date - pd.Timedelta(days=period)
        preds.append(history.loc[candidate] if candidate in history.index else history.tail(period).mean())
    return pd.Series(preds, index=horizon_index)


## 5. Baselines

In [23]:
for group in GROUPS_TO_FORECAST:
    print(f'\n{"="*80}\nBASELINES â€” {group}\n{"="*80}')
    train, val, test = series_data[group]['train'], series_data[group]['val'], series_data[group]['test']
    train_val = pd.concat([train, val])
    y_train_val = train_val[TARGET_COL]
    y_test = test[TARGET_COL]

    naive_pred = pd.Series(y_train_val.iloc[-1], index=y_test.index)
    seasonal_pred = seasonal_naive_forecast(y_train_val, y_test.index, period=FORECAST_SEASONAL_PERIOD)
    rolling7_pred = pd.Series(y_train_val.tail(7).mean(), index=y_test.index)

    rows = [
        save_result(group, 'Naive', y_test, naive_pred),
        save_result(group, f'Seasonal Naive ({FORECAST_SEASONAL_PERIOD}d)', y_test, seasonal_pred),
        save_result(group, 'Rolling Mean (7d)', y_test, rolling7_pred),
    ]

    X_train_val = train_val[feature_cols].astype(float).fillna(0)
    X_test = test[feature_cols].astype(float).fillna(0)

    ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
    ridge.fit(X_train_val, y_train_val)
    rows.append(save_result(group, 'Ridge Features', y_test, pd.Series(ridge.predict(X_test), index=y_test.index)))

    rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    rf.fit(X_train_val, y_train_val)
    rows.append(save_result(group, 'Random Forest Features', y_test, pd.Series(rf.predict(X_test), index=y_test.index)))

    display(pd.DataFrame(rows).set_index('Model').sort_values('RMSE').round(4))



BASELINES â€” Vitoria-Gasteiz


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Ridge Features,0.0150,0.0210,5.1256,5.1276
Random Forest Features,0.0228,0.0264,7.3459,7.6782
Seasonal Naive (365d),0.0344,0.0450,12.3026,11.7859
Naive,0.0986,0.1149,31.2970,38.6875
Rolling Mean (7d),0.1045,0.1205,33.3135,41.6463



BASELINES â€” Donostia/San Sebastian


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Ridge Features,0.0139,0.0193,3.5305,3.6175
Random Forest Features,0.0228,0.0267,5.7807,6.0050
Seasonal Naive (365d),0.0400,0.0523,10.6751,10.3150
Rolling Mean (7d),0.0739,0.0938,17.8335,20.4680
Naive,0.0912,0.1090,22.4008,26.2052



BASELINES â€” Pamplona/Iruna


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Random Forest Features,0.0113,0.0139,3.1207,3.1852
Ridge Features,0.0136,0.0171,3.8770,3.9074
Seasonal Naive (365d),0.0338,0.0473,9.2502,9.7151
Naive,0.0637,0.0820,16.4780,18.7399
Rolling Mean (7d),0.0818,0.0985,21.5065,25.0787


## 6. SARIMA and SARIMAX

In [24]:
if RUN_SARIMA:
    SARIMA_ORDERS = [(1, 1, 1), (2, 1, 1), (1, 1, 2)]
    SARIMA_SEASONAL_ORDERS = [(1, 1, 1, SARIMA_SEASONAL_PERIOD), (0, 1, 1, SARIMA_SEASONAL_PERIOD)]

    for group in GROUPS_TO_FORECAST:
        print(f'\n{"="*80}\nSARIMA â€” {group}\n{"="*80}')
        train, val, test = series_data[group]['train'], series_data[group]['val'], series_data[group]['test']
        train_val = pd.concat([train, val])
        y_train_val, y_test = train_val[TARGET_COL].astype(float), test[TARGET_COL].astype(float)

        # v3: log-space for near-zero series (prevents MAPE explosion and stabilises differencing)
        use_log = float(y_train_val.min()) < 0.5
        if use_log:
            print(f'  [log-space] min={y_train_val.min():.4f} â†’ log1p applied')
            y_fit = np.log1p(y_train_val.clip(lower=0))
        else:
            y_fit = y_train_val

        best = {'rmse': np.inf, 'pred': None, 'params': None}

        for order in SARIMA_ORDERS:
            for s_order in SARIMA_SEASONAL_ORDERS:
                try:
                    model = SARIMAX(y_fit, order=order, seasonal_order=s_order,
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fitted = model.fit(disp=False, maxiter=150, low_memory=True)
                    raw_pred = pd.Series(fitted.forecast(steps=len(y_test)).values, index=y_test.index)
                    pred = np.expm1(raw_pred).clip(lower=0) if use_log else raw_pred
                    rmse = np.sqrt(mean_squared_error(y_test, pred))
                    if rmse < best['rmse']:
                        best = {'rmse': rmse, 'pred': pred, 'params': (order, s_order)}
                except Exception as e:
                    print(f'Failed {order} {s_order}: {e}')

        if best['pred'] is not None:
            metrics = save_result(group, 'SARIMA', y_test, best['pred'], {'params': best['params'], 'log_transform': use_log})
            print('Best params:', best['params'])
            display(pd.DataFrame([metrics]).set_index('Model').round(4))

if RUN_SARIMAX:
    SARIMAX_EXOG_COLS = [c for c in [
        'is_weekend', 'day_of_week', 'month', 'quarter', 'week_of_year',
        'is_holiday_es', 'is_bridge_day', 'lag_1d', 'lag_7d', 'roll7_mean',
        'roll30_mean', 'wow_change', 'dod_change', 'zscore_municipality', 'pct_rank_global'
    ] if c in feature_cols]

    for group in GROUPS_TO_FORECAST:
        print(f'\n{"="*80}\nSARIMAX â€” {group}\n{"="*80}')
        train, val, test = series_data[group]['train'], series_data[group]['val'], series_data[group]['test']
        train_val = pd.concat([train, val])
        y_train_val, y_test = train_val[TARGET_COL].astype(float), test[TARGET_COL].astype(float)
        X_train_val = train_val[SARIMAX_EXOG_COLS].astype(float).fillna(0)
        X_test = test[SARIMAX_EXOG_COLS].astype(float).fillna(0)

        # v3: log-space for near-zero series
        use_log = float(y_train_val.min()) < 0.5
        if use_log:
            print(f'  [log-space] min={y_train_val.min():.4f} â†’ log1p applied')
            y_fit = np.log1p(y_train_val.clip(lower=0))
        else:
            y_fit = y_train_val

        try:
            model = SARIMAX(y_fit, exog=X_train_val, order=(1,1,1),
                            seasonal_order=(1,1,1,SARIMA_SEASONAL_PERIOD),
                            enforce_stationarity=False, enforce_invertibility=False)
            fitted = model.fit(disp=False, maxiter=150, low_memory=True)
            raw_pred = pd.Series(fitted.forecast(steps=len(y_test), exog=X_test).values, index=y_test.index)
            pred = np.expm1(raw_pred).clip(lower=0) if use_log else raw_pred
            metrics = save_result(group, 'SARIMAX', y_test, pred, {'exog_cols': SARIMAX_EXOG_COLS, 'log_transform': use_log})
            display(pd.DataFrame([metrics]).set_index('Model').round(4))
        except Exception as e:
            print('SARIMAX failed:', e)



SARIMA â€” Vitoria-Gasteiz
  [log-space] min=0.1012 â†’ log1p applied
Failed (2, 1, 1) (0, 1, 1, 7): Input contains infinity or a value too large for dtype('float64').
Best params: ((1, 1, 1), (1, 1, 1, 7))


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMA,0.2214,0.2392,74.8513,132.3827



SARIMA â€” Donostia/San Sebastian
  [log-space] min=0.0261 â†’ log1p applied
Best params: ((1, 1, 1), (1, 1, 1, 7))


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMA,0.0541,0.0732,12.9825,14.3795



SARIMA â€” Pamplona/Iruna
  [log-space] min=0.1638 â†’ log1p applied
Best params: ((1, 1, 1), (1, 1, 1, 7))


...

,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMAX,0.1557,0.1691,52.9898,77.5525



SARIMAX â€” Donostia/San Sebastian
  [log-space] min=0.0261 â†’ log1p applied


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMAX,0.0109,0.015,3.1526,3.0629



SARIMAX â€” Pamplona/Iruna
  [log-space] min=0.1638 â†’ log1p applied


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMAX,0.023,0.0265,6.2338,6.4834


## 7. PyTorch utilities

In [25]:
# â”€â”€ Datasets â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class WindowDataset(Dataset):
    """Single-stream: (seq_len, n_features) â†’ scalar."""
    def __init__(self, X, y, seq_length=30):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
        self.seq_length = seq_length
    def __len__(self): return len(self.y) - self.seq_length
    def __getitem__(self, idx):
        return self.X[idx:idx+self.seq_length], self.y[idx+self.seq_length]

class DualWindowDataset(Dataset):
    """
    v2 â€” Dual-stream dataset for the improved LSTM.
    Returns:
      x_seq    : (seq_len, n_seq_feats)  â€” dynamic features over lookback window
      x_static : (n_static_feats,)       â€” calendar features for the PREDICTION day
      y        : scalar target
    """
    def __init__(self, X_seq, X_static, y, seq_length=30):
        self.X_seq    = torch.tensor(X_seq,    dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y        = torch.tensor(y,        dtype=torch.float32).view(-1, 1)
        self.seq_length = seq_length
    def __len__(self): return len(self.y) - self.seq_length
    def __getitem__(self, idx):
        return (
            self.X_seq   [idx : idx + self.seq_length],
            self.X_static[idx + self.seq_length],
            self.y       [idx + self.seq_length],
        )

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# â”€â”€ Scalers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.preprocessing import StandardScaler

def make_scaled_xy(train_df, val_df, test_df, cols, target_col):
    """Single-stream scaler (used by N-BEATS and baselines)."""
    x_sc, y_sc = StandardScaler(), StandardScaler()
    X_tr = x_sc.fit_transform(train_df[cols].astype(float).fillna(0))
    X_va = x_sc.transform(val_df[cols].astype(float).fillna(0))
    X_te = x_sc.transform(test_df[cols].astype(float).fillna(0))
    y_tr = y_sc.fit_transform(train_df[[target_col]].astype(float)).ravel()
    y_va = y_sc.transform(val_df[[target_col]].astype(float)).ravel()
    y_te = y_sc.transform(test_df[[target_col]].astype(float)).ravel()
    return X_tr, y_tr, X_va, y_va, X_te, y_te, x_sc, y_sc

def make_dual_scaled_xy(train_df, val_df, test_df, seq_cols, static_cols, target_col):
    """
    v2 â€” Dual-stream scaler for the improved LSTM.
    seq_cols    : dynamic features (history/lags) fed step-by-step into LSTM
    static_cols : calendar features for the prediction day
    """
    xs_sc, xc_sc, y_sc = StandardScaler(), StandardScaler(), StandardScaler()

    Xs_tr = xs_sc.fit_transform(train_df[seq_cols].astype(float).fillna(0))
    Xs_va = xs_sc.transform(val_df[seq_cols].astype(float).fillna(0))
    Xs_te = xs_sc.transform(test_df[seq_cols].astype(float).fillna(0))

    Xc_tr = xc_sc.fit_transform(train_df[static_cols].astype(float).fillna(0))
    Xc_va = xc_sc.transform(val_df[static_cols].astype(float).fillna(0))
    Xc_te = xc_sc.transform(test_df[static_cols].astype(float).fillna(0))

    y_tr = y_sc.fit_transform(train_df[[target_col]].astype(float)).ravel()
    y_va = y_sc.transform(val_df[[target_col]].astype(float)).ravel()
    y_te = y_sc.transform(test_df[[target_col]].astype(float)).ravel()

    return Xs_tr, Xc_tr, y_tr, Xs_va, Xc_va, y_va, Xs_te, Xc_te, y_te, xs_sc, xc_sc, y_sc


# â”€â”€ Generic single-stream training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def train_torch_model(model, train_loader, val_loader, epochs=80, lr=1e-3, patience=12):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_state, best_val, bad = None, np.inf, 0
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); losses.append(loss.item())
        model.eval(); vloss = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vloss.append(loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).item())
        vl = float(np.mean(vloss)) if vloss else np.inf
        if vl < best_val:
            best_val = vl
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d} | train={np.mean(losses):.5f} | val={vl:.5f}')
        if bad >= patience:
            print(f'Early stop at epoch {epoch}. Best val={best_val:.5f}'); break
    if best_state: model.load_state_dict(best_state)
    return model


# â”€â”€ v2: Dual-stream training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def train_dual_model(model, train_loader, val_loader, epochs=80, lr=1e-3, patience=12):
    """Training loop for models with signature model(x_seq, x_static) â†’ y."""
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_state, best_val, bad = None, np.inf, 0
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for x_seq, x_static, yb in train_loader:
            x_seq, x_static, yb = x_seq.to(DEVICE), x_static.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x_seq, x_static), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); losses.append(loss.item())
        model.eval(); vloss = []
        with torch.no_grad():
            for x_seq, x_static, yb in val_loader:
                vloss.append(loss_fn(model(x_seq.to(DEVICE), x_static.to(DEVICE)), yb.to(DEVICE)).item())
        vl = float(np.mean(vloss)) if vloss else np.inf
        if vl < best_val:
            best_val = vl
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d} | train={np.mean(losses):.5f} | val={vl:.5f}')
        if bad >= patience:
            print(f'Early stop at epoch {epoch}. Best val={best_val:.5f}'); break
    if best_state: model.load_state_dict(best_state)
    return model


# â”€â”€ Prediction helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def predict_windows(model, X_context, y_scaler, index, seq_length=30):
    """Single-stream rolling predictions."""
    model.eval()
    ds     = WindowDataset(X_context, np.zeros(len(X_context)), seq_length)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    preds  = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    preds = y_scaler.inverse_transform(np.vstack(preds).ravel().reshape(-1,1)).ravel()
    return pd.Series(preds, index=index[seq_length:])

def predict_dual_windows(model, Xs_context, Xc_context, y_scaler, index, seq_length=30):
    """
    v2 â€” Dual-stream rolling predictions.
    Xs_context : sequential features for the full context (val+test)
    Xc_context : calendar features for the full context (val+test)
    """
    model.eval()
    ds     = DualWindowDataset(Xs_context, Xc_context, np.zeros(len(Xs_context)), seq_length)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    preds  = []
    with torch.no_grad():
        for x_seq, x_static, _ in loader:
            preds.append(model(x_seq.to(DEVICE), x_static.to(DEVICE)).cpu().numpy())
    preds = y_scaler.inverse_transform(np.vstack(preds).ravel().reshape(-1,1)).ravel()
    return pd.Series(preds, index=index[seq_length:])


## 8. LSTM v2 â€” Dual-Stream Architecture

**Problem with v1:** all features (both calendar and lag) were fed into the same
LSTM stream. Calendar features like `is_holiday_es` are *static for the prediction day*
â€” passing them as sequential inputs over the lookback dilutes their signal.

**v2 fix â€” two streams:**

```
Lookback window (seq_length Ã— |HISTORY_FEATURES|)
      â””â”€â–º LSTM (2 layers, hidden=64)
                â””â”€â–º last hidden state â”€â”€â”
                                        â”œâ”€â–º Linear head â†’ Å·
Prediction day (|CALENDAR_FEATURES|)   â”‚
      â””â”€â–º Static encoder (Linear+LN) â”€â”€â”˜
```

The **LSTM stream** sees dynamic features (lags, rolling stats) that change each day.  
The **static stream** sees calendar features for the day we're predicting â€” it can tell
the LSTM "tomorrow is a holiday" without that information being buried in the sequence.


In [26]:
class LSTMForecaster(nn.Module):
    """
    v2 â€” Dual-stream LSTM.
    Sequential stream : lookback window of dynamic (lag/rolling) features
    Static stream     : calendar features for the prediction day (known in advance)
    """
    def __init__(self, seq_input_size, static_size, hidden=48, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            seq_input_size, hidden, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.static_enc = nn.Sequential(
            nn.Linear(static_size, 32),
            nn.ReLU(),
            nn.LayerNorm(32),          # stabilises scale differences between calendar features
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_static):
        lstm_out, _ = self.lstm(x_seq)            # (B, seq_len, hidden)
        static_enc  = self.static_enc(x_static)   # (B, 32)
        combined    = torch.cat([lstm_out[:, -1, :], static_enc], dim=1)
        return self.head(combined)


if RUN_LSTM:
    for group in GROUPS_TO_FORECAST:
        print(f'\n{"="*80}\nLSTM v2 (dual-stream) â€” {group}\n{"="*80}')
        train, val, test = series_data[group]['train'], series_data[group]['val'], series_data[group]['test']

        # â”€â”€ scale with the safe NEURAL feature split â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        seq_feats    = [f for f in HISTORY_FEATURES  if f in train.columns]
        static_feats = [f for f in CALENDAR_FEATURES if f in train.columns]

        (Xs_tr, Xc_tr, y_tr,
         Xs_va, Xc_va, y_va,
         Xs_te, Xc_te, y_te,
         _, _, y_scaler) = make_dual_scaled_xy(train, val, test, seq_feats, static_feats, TARGET_COL)

        train_loader = DataLoader(
            DualWindowDataset(Xs_tr, Xc_tr, y_tr, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(
            DualWindowDataset(Xs_va, Xc_va, y_va, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

        model = LSTMForecaster(
            seq_input_size = len(seq_feats),
            static_size    = len(static_feats),
        )
        model = train_dual_model(model, train_loader, val_loader, EPOCHS, patience=PATIENCE)

        # context = val + test to seed the window
        Xs_ctx = np.vstack([Xs_va, Xs_te])
        Xc_ctx = np.vstack([Xc_va, Xc_te])
        ctx_idx = val.index.append(test.index)

        pred    = predict_dual_windows(model, Xs_ctx, Xc_ctx, y_scaler, ctx_idx, SEQ_LENGTH)
        pred    = pred.reindex(test.index).dropna()
        actual  = test.loc[pred.index, TARGET_COL]
        metrics = save_result(group, 'LSTM v2 (dual-stream)', actual, pred)
        display(pd.DataFrame([metrics]).set_index('Model').round(4))
        gc.collect()



LSTM v2 (dual-stream) â€” Vitoria-Gasteiz
Epoch 001 | train=0.75926 | val=2.97892
Epoch 010 | train=0.13403 | val=0.99090
Epoch 020 | train=0.11953 | val=0.94990
Early stop at epoch 27. Best val=0.72708


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
LSTM v2 (dual-stream),0.0296,0.0392,9.6673,9.985



LSTM v2 (dual-stream) â€” Donostia/San Sebastian
Epoch 001 | train=0.80156 | val=1.16421
Epoch 010 | train=0.11492 | val=0.19520
Epoch 020 | train=0.09601 | val=0.25478
Early stop at epoch 24. Best val=0.18605


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
LSTM v2 (dual-stream),0.0244,0.0328,6.0757,6.2644



LSTM v2 (dual-stream) â€” Pamplona/Iruna
Epoch 001 | train=0.81813 | val=0.39974
Epoch 010 | train=0.15227 | val=0.14085
Epoch 020 | train=0.13338 | val=0.12744
Epoch 030 | train=0.11737 | val=0.15141
Early stop at epoch 35. Best val=0.12744


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
LSTM v2 (dual-stream),0.024,0.0305,6.8237,6.8763


## 9. N-BEATS v2 â€” Covariate-Conditioned Blocks

**Problem with v1:** the block forward pass used `x[:, :, 0]` â€” only the target
column. All other features were silently ignored even though they were passed in.

**v2 fix:** at each block, the last-step calendar + lag features are concatenated
to the residual before entering the FC tower. This gives each block a conditioning
signal: *"this window ends on a holiday"* changes how it decomposes the signal.

```
residual_t  (seq_len,)
covariates  (|NEURAL_FEATURES|,) â† last step's features
    â””â”€â–º cat([residual_t, covariates])
             â””â”€â–º FC tower â†’ backcast + forecast
```


In [27]:
class NBeatsBlock(nn.Module):
    """
    v2 â€” Generic N-BEATS block with optional covariate conditioning.
    The covariate vector (last-step features) is concatenated to the residual
    before the shared FC tower, so every block can condition on known context.
    """
    def __init__(self, input_size, hidden_size=64, n_layers=4, covariate_size=0, dropout=0.2):
        super().__init__()
        fc_in = input_size + covariate_size
        layers, n = [], fc_in
        for _ in range(n_layers):
            layers += [nn.Linear(n, hidden_size), nn.ReLU(), nn.Dropout(dropout)]
            n = hidden_size
        self.net      = nn.Sequential(*layers)
        self.backcast = nn.Linear(hidden_size, input_size)
        self.forecast = nn.Linear(hidden_size, 1)

    def forward(self, residual, covariates=None):
        x = torch.cat([residual, covariates], dim=-1) if covariates is not None else residual
        h = self.net(x)
        return self.backcast(h), self.forecast(h)


class SimpleNBeats(nn.Module):
    """
    v2 â€” N-BEATS with covariate conditioning.
    Input tensor layout (set by make_scaled_xy with nbeats_cols):
      column 0      : target (normalised avg_kwh)
      columns 1..N  : NEURAL_FEATURES (lags + calendar)

    The target column drives the residual chain; the feature columns
    are extracted at the last time-step and used as a conditioning vector.
    """
    def __init__(self, input_size=30, n_blocks=3, hidden_size=64, covariate_size=0, dropout=0.2):
        super().__init__()
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, hidden_size, covariate_size=covariate_size, dropout=dropout)
            for _ in range(n_blocks)
        ])

    def forward(self, x):
        # x: (B, seq_len, n_cols)
        residual   = x[:, :, 0]                    # target channel â€” (B, seq_len)
        covariates = x[:, -1, 1:] if x.size(2) > 1 else None   # last-step features â€” (B, n_feats)
        forecast   = torch.zeros(x.size(0), 1, device=x.device)
        for block in self.blocks:
            backcast, block_fc = block(residual, covariates)
            residual  = residual - backcast
            forecast  = forecast + block_fc
        return forecast


if RUN_NBEATS:
    for group in GROUPS_TO_FORECAST:
        print(f'\n{"="*80}\nN-BEATS v2 (covariate-conditioned) â€” {group}\n{"="*80}')
        train, val, test = series_data[group]['train'].copy(), series_data[group]['val'].copy(), series_data[group]['test'].copy()

        # Target first, then safe neural features
        nb_neural    = [f for f in NEURAL_FEATURES if f in train.columns]
        nbeats_cols  = [TARGET_COL] + nb_neural
        nbeats_cols  = list(dict.fromkeys(nbeats_cols))

        X_tr, y_tr, X_va, y_va, X_te, _, _, y_scaler = make_scaled_xy(train, val, test, nbeats_cols, TARGET_COL)

        train_loader = DataLoader(WindowDataset(X_tr, y_tr, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(WindowDataset(X_va, y_va, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

        covariate_size = len(nb_neural)   # features beyond the target column
        model = SimpleNBeats(
            input_size     = SEQ_LENGTH,
            n_blocks       = 3,
            hidden_size    = 64,
            covariate_size = covariate_size,
        )
        model = train_torch_model(model, train_loader, val_loader, EPOCHS, patience=PATIENCE)

        X_ctx   = np.vstack([X_va, X_te])
        ctx_idx = val.index.append(test.index)
        pred    = predict_windows(model, X_ctx, y_scaler, ctx_idx, SEQ_LENGTH).reindex(test.index).dropna()
        actual  = test.loc[pred.index, TARGET_COL]
        metrics = save_result(group, 'N-BEATS v2 (covariate-conditioned)', actual, pred)
        display(pd.DataFrame([metrics]).set_index('Model').round(4))
        gc.collect()



N-BEATS v2 (covariate-conditioned) â€” Vitoria-Gasteiz
Epoch 001 | train=0.50281 | val=0.66867
Epoch 010 | train=0.07686 | val=0.27232
Epoch 020 | train=0.05776 | val=0.27319
Epoch 030 | train=0.05101 | val=0.26056
Epoch 040 | train=0.04569 | val=0.23068
Early stop at epoch 41. Best val=0.21577


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
N-BEATS v2 (covariate-conditioned),0.0154,0.0207,5.3093,5.3047



N-BEATS v2 (covariate-conditioned) â€” Donostia/San Sebastian
Epoch 001 | train=0.52817 | val=0.26775
Epoch 010 | train=0.07729 | val=0.12139
Epoch 020 | train=0.05552 | val=0.08354
Epoch 030 | train=0.04581 | val=0.05550
Epoch 040 | train=0.03872 | val=0.06283
Epoch 050 | train=0.03557 | val=0.06359
Epoch 060 | train=0.03415 | val=0.07057
Early stop at epoch 63. Best val=0.05358


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
N-BEATS v2 (covariate-conditioned),0.0132,0.0183,3.3803,3.4003



N-BEATS v2 (covariate-conditioned) â€” Pamplona/Iruna
Epoch 001 | train=0.64856 | val=0.36678
Epoch 010 | train=0.09819 | val=0.09874
Epoch 020 | train=0.07608 | val=0.09488
Epoch 030 | train=0.06261 | val=0.08665
Epoch 040 | train=0.05454 | val=0.08714
Epoch 050 | train=0.04884 | val=0.08489
Epoch 060 | train=0.04431 | val=0.10487
Early stop at epoch 60. Best val=0.08390


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
N-BEATS v2 (covariate-conditioned),0.0159,0.0227,4.4211,4.4406


## 10. Proposed hybrid: Seasonal Naive + Neural Network residual

Formula:

`forecast_t = seasonal_naive_t + residual_nn(features_t)`

The seasonal naive handles the repeated weekly/yearly pattern. The neural network only learns the residual/error left by that baseline.

### Hybrid v2 â€” ResidualLSTM

**Problem with v1:** `ResidualMLP` was a *tabular* model â€” it took a single-day
snapshot of features to predict the residual. It had no temporal memory.

**v2 fix:** replace the MLP with a **ResidualLSTM** that sees a lookback window
where each step is `[normalised_residual_t, calendar_features_t, lag_features_t]`.
This gives the corrector the same kind of sequential context the main LSTM has,
but operating on the *error signal* rather than the raw target.

```
Window of past (residual + features):
  [r_{t-28}, feat_{t-28}]
  [r_{t-27}, feat_{t-27}]
       ...
  [r_{t-1},  feat_{t-1}]
        â””â”€â–º LSTM â”€â”€â–º corrected residual Ãª_t

Å·_hybrid(t) = Å·_SN(t) + Ãª_t
```


In [28]:
class ResidualLSTM(nn.Module):
    """
    v2 â€” Sequential residual corrector.
    Each time-step input: [normalised_residual_t, calendar_features_t, lag_features_t]
    Output: predicted residual for the next step.
    """
    def __init__(self, input_size, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


if RUN_HYBRID:
    for group in GROUPS_TO_FORECAST:
        print(f'\n{"="*80}\nHYBRID v2 â€” Seasonal Naive + ResidualLSTM â€” {group}\n{"="*80}')
        train, val, test = series_data[group]['train'], series_data[group]['val'], series_data[group]['test']
        train_val   = pd.concat([train, val])
        y_train_val = train_val[TARGET_COL].astype(float)

        # â”€â”€ Step 1: Seasonal Naive component â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        sn_fitted  = y_train_val.shift(FORECAST_SEASONAL_PERIOD).fillna(y_train_val.expanding().mean())
        residuals  = y_train_val - sn_fitted

        # â”€â”€ Step 2: Build sequential residual + feature matrix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        hybrid_feats = [f for f in NEURAL_FEATURES if f in train.columns]

        def build_residual_matrix(df_slice, residual_series, feat_cols, r_scaler=None, f_scaler=None):
            """
            Returns scaled matrix where column 0 = normalised residual,
            columns 1.. = normalised neural features.
            """
            res_vals  = residual_series.loc[df_slice.index].values.reshape(-1,1)
            feat_vals = df_slice[feat_cols].astype(float).fillna(0).values
            if r_scaler is None:
                r_scaler = StandardScaler().fit(res_vals)
            if f_scaler is None:
                f_scaler = StandardScaler().fit(feat_vals)
            return (np.hstack([r_scaler.transform(res_vals),
                               f_scaler.transform(feat_vals)]),
                    r_scaler, f_scaler)

        X_res_tr, r_sc, f_sc = build_residual_matrix(train, residuals, hybrid_feats)
        X_res_va, _,    _    = build_residual_matrix(val,   residuals, hybrid_feats, r_sc, f_sc)

        # normalised residuals as targets
        y_res_tr = r_sc.transform(residuals.loc[train.index].values.reshape(-1,1)).ravel()
        y_res_va = r_sc.transform(residuals.loc[val.index].values.reshape(-1,1)).ravel()

        train_loader = DataLoader(WindowDataset(X_res_tr, y_res_tr, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(WindowDataset(X_res_va, y_res_va, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

        res_lstm_input_size = 1 + len(hybrid_feats)   # residual col + feature cols
        model = ResidualLSTM(input_size=res_lstm_input_size, hidden=32, n_layers=1, dropout=0.3)
        print(f'ResidualLSTM params: {sum(p.numel() for p in model.parameters()):,}')
        model = train_torch_model(model, train_loader, val_loader, EPOCHS, patience=PATIENCE)

        # â”€â”€ Step 3: Predict residuals on test period â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        # Context seed = last SEQ_LENGTH rows of val residual matrix
        X_res_seed = X_res_va[-SEQ_LENGTH:]

        # Build test feature matrix (calendar + lags known for test days)
        X_test_feats = test[hybrid_feats].astype(float).fillna(0).values
        X_test_feats = f_sc.transform(X_test_feats)
        # residual placeholder for test (we don't know them; set to 0 â€” will be updated step by step)
        res_preds_norm = []
        buffer = list(X_res_seed)

        model.eval()
        with torch.no_grad():
            for i in range(len(test)):
                window = np.array(buffer[-SEQ_LENGTH:])          # (SEQ_LENGTH, 1 + n_feats)
                x = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                p = model(x).item()
                res_preds_norm.append(p)
                # next step: [predicted_residual, true test features for that day]
                next_row = np.hstack([[p], X_test_feats[i]])
                buffer.append(next_row)

        res_pred = r_sc.inverse_transform(np.array(res_preds_norm).reshape(-1,1)).ravel()

        # â”€â”€ Step 4: Final hybrid = SN + residual correction â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        sn_test = seasonal_naive_forecast(y_train_val, test.index, period=FORECAST_SEASONAL_PERIOD)
        hybrid_pred = pd.Series(sn_test.values + res_pred, index=test.index)

        metrics = save_result(group,
                              f'Hybrid v2 SeasonalNaive({FORECAST_SEASONAL_PERIOD}d)+ResidualLSTM',
                              test[TARGET_COL], hybrid_pred)
        display(pd.DataFrame([metrics]).set_index('Model').round(4))
        gc.collect()



HYBRID v2 â€” Seasonal Naive + ResidualLSTM â€” Vitoria-Gasteiz
ResidualLSTM params: 7,233
Epoch 001 | train=0.92180 | val=1.72933
Epoch 010 | train=0.19644 | val=0.59181
Epoch 020 | train=0.17394 | val=0.55577
Epoch 030 | train=0.14034 | val=0.50889
Early stop at epoch 34. Best val=0.48487


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Hybrid v2 SeasonalNaive(365d)+ResidualLSTM,0.064,0.0778,24.3009,20.4983



HYBRID v2 â€” Seasonal Naive + ResidualLSTM â€” Donostia/San Sebastian
ResidualLSTM params: 7,233
Epoch 001 | train=0.86614 | val=0.43971
Epoch 010 | train=0.13214 | val=0.11138
Epoch 020 | train=0.10490 | val=0.11200
Early stop at epoch 27. Best val=0.10528


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Hybrid v2 SeasonalNaive(365d)+ResidualLSTM,0.1357,0.1557,37.7725,30.3791



HYBRID v2 â€” Seasonal Naive + ResidualLSTM â€” Pamplona/Iruna
ResidualLSTM params: 7,233
Epoch 001 | train=0.96292 | val=0.42962
Epoch 010 | train=0.21425 | val=0.27552
Epoch 020 | train=0.16847 | val=0.23359
Epoch 030 | train=0.13660 | val=0.19614
Epoch 040 | train=0.12471 | val=0.18161
Epoch 050 | train=0.11334 | val=0.17226
Epoch 060 | train=0.09666 | val=0.17614
Early stop at epoch 60. Best val=0.17049


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Hybrid v2 SeasonalNaive(365d)+ResidualLSTM,0.0823,0.1049,22.0206,26.1121


## 11. Final comparison and plots

In [29]:
comparison_dfs = {}
summary_rows = []

for group in GROUPS_TO_FORECAST:
    comp = pd.DataFrame([v['metrics'] for v in all_results[group].values()]).set_index('Model').sort_values('RMSE')
    comparison_dfs[group] = comp
    print(f'\n{"="*100}\nFINAL COMPARISON â€” {group}\n{"="*100}')
    display(comp.round(4))

    best = comp.iloc[0]
    baseline_names = [i for i in comp.index if i == 'Naive' or i.startswith('Seasonal Naive') or i.startswith('Rolling')]
    best_baseline = comp.loc[baseline_names].sort_values('RMSE').iloc[0]
    summary_rows.append({
        GROUP_COL: group,
        'Best model': best.name,
        'Best RMSE': best['RMSE'],
        'Best baseline': best_baseline.name,
        'Baseline RMSE': best_baseline['RMSE'],
        'Improvement vs baseline (%)': (best_baseline['RMSE'] - best['RMSE']) / best_baseline['RMSE'] * 100
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(4))

for group in GROUPS_TO_FORECAST:
    test = series_data[group]['test'][TARGET_COL]
    best_model = comparison_dfs[group].index[0]
    plt.figure(figsize=(16, 6))
    plt.plot(test.index, test.values, label='Actual', linewidth=2)
    plt.plot(all_results[group][best_model]['pred'].index, all_results[group][best_model]['pred'].values, label=f'Best: {best_model}', linewidth=2)
    for name, data in all_results[group].items():
        if name.startswith('Seasonal Naive') or name.startswith('Hybrid'):
            plt.plot(data['pred'].index, data['pred'].values, label=name, alpha=0.75)
    plt.title(f'{group} â€” Forecast comparison')
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()



FINAL COMPARISON â€” Vitoria-Gasteiz


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
N-BEATS v2 (covariate-conditioned),0.0154,0.0207,5.3093,5.3047
Ridge Features,0.0150,0.0210,5.1256,5.1276
Random Forest Features,0.0228,0.0264,7.3459,7.6782
LSTM v2 (dual-stream),0.0296,0.0392,9.6673,9.9850
Seasonal Naive (365d),0.0344,0.0450,12.3026,11.7859
Hybrid v2 SeasonalNaive(365d)+ResidualLSTM,0.0640,0.0778,24.3009,20.4983
Naive,0.0986,0.1149,31.2970,38.6875
Rolling Mean (7d),0.1045,0.1205,33.3135,41.6463
SARIMAX,0.1557,0.1691,52.9898,77.5525



FINAL COMPARISON â€” Donostia/San Sebastian


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
SARIMAX,0.0109,0.0150,3.1526,3.0629
N-BEATS v2 (covariate-conditioned),0.0132,0.0183,3.3803,3.4003
Ridge Features,0.0139,0.0193,3.5305,3.6175
Random Forest Features,0.0228,0.0267,5.7807,6.0050
LSTM v2 (dual-stream),0.0244,0.0328,6.0757,6.2644
Seasonal Naive (365d),0.0400,0.0523,10.6751,10.3150
SARIMA,0.0541,0.0732,12.9825,14.3795
Rolling Mean (7d),0.0739,0.0938,17.8335,20.4680
Naive,0.0912,0.1090,22.4008,26.2052



FINAL COMPARISON â€” Pamplona/Iruna


,MAE,RMSE,MAPE(%),sMAPE(%)
Model,,,,
Random Forest Features,0.0113,0.0139,3.1207,3.1852
Ridge Features,0.0136,0.0171,3.8770,3.9074
N-BEATS v2 (covariate-conditioned),0.0159,0.0227,4.4211,4.4406
SARIMAX,0.0230,0.0265,6.2338,6.4834
LSTM v2 (dual-stream),0.0240,0.0305,6.8237,6.8763
Seasonal Naive (365d),0.0338,0.0473,9.2502,9.7151
SARIMA,0.0522,0.0663,14.9218,14.6424
Naive,0.0637,0.0820,16.4780,18.7399
Rolling Mean (7d),0.0818,0.0985,21.5065,25.0787


,municipality,Best model,Best RMSE,Best baseline,Baseline RMSE,Improvement vs baseline (%)
0,Vitoria-Gasteiz,N-BEATS v2 (covariate-conditioned),0.0207,Seasonal Naive (365d),0.0450,54.0857
1,Donostia/San Sebastian,SARIMAX,0.0150,Seasonal Naive (365d),0.0523,71.4096
2,Pamplona/Iruna,Random Forest Features,0.0139,Seasonal Naive (365d),0.0473,70.6983


## 12. Save outputs

In [30]:
OUTPUT_DIR = BASE_PATH / f'forecasting_model_results_{FORECAST_SEASONAL_PERIOD}_seasonality_v4_{TARGET_COL}'
FORECAST_VAR = TARGET_COL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / 'summary_results.csv'
summary_df.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)

for group, comp in comparison_dfs.items():
    safe = str(group).replace('/', '_').replace(' ', '_')
    comp.to_csv(OUTPUT_DIR / f'{safe}_metrics.csv')
    pred_df = pd.DataFrame(index=series_data[group]['test'].index)
    pred_df['actual'] = series_data[group]['test'][TARGET_COL]
    for model_name, data in all_results[group].items():
        pred_df[model_name] = data['pred']
    pred_df.to_csv(OUTPUT_DIR / f'{safe}_predictions.csv')
print('Done.')

Saved summary: C:\Users\GONCA\Desktop\Iscte\MCD\Theses\forecasting_model_results_365_seasonality_v4_avg_kwh\summary_results.csv
Done.


> Thesis text suggestion: The hybrid architecture decomposes the forecasting problem into a deterministic seasonal component and a non-linear residual component. The seasonal component is estimated with a seasonal naive rule, which is a strong baseline for daily electricity consumption because it directly preserves repeated demand patterns. The residual series, defined as the difference between observed consumption and the seasonal naive estimate, is then modelled using a neural network with calendar, lag, rolling, holiday, bridge-day, and municipality-normalised features. The final forecast is obtained by adding the predicted residual to the seasonal naive component.

## 13. Target Variable Comparison: `avg_kwh` vs `avg_kwh_per_user`

`avg_kwh` is the raw daily average kWh aggregated across all smart meters in the municipality.  
`avg_kwh_per_user` divides by the number of active users (`avg_kwh / n_users`), controlling for the fact that (a) cities differ in size and (b) user counts grow over time as the cooperative expands.

**Why this comparison matters:**  
If user count is correlated with the target (e.g. the cooperative onboards new users mid-series), `avg_kwh` will show a spurious upward trend that the models must learn to ignore. `avg_kwh_per_user` removes this confounder and makes the three municipalities directly comparable on a per-capita basis.

**How to generate the `avg_kwh_per_user` results:**  
1. Change `TARGET_CANDIDATES = ['avg_kwh']` â†’ `TARGET_CANDIDATES = ['avg_kwh_per_user']` in cell 0 (config).  
2. Re-run all cells (Kernel â†’ Restart & Run All).  
3. Results will be saved automatically to `forecasting_model_results_365_seasonality_v4_avg_kwh_per_user/`.  
4. Restore `TARGET_CANDIDATES = ['avg_kwh']` and re-run if you want both result sets for the comparison below.

MAPE and sMAPE are used for the comparison because the two targets have different absolute scales (MAE/RMSE are not directly comparable).  
**Negative delta** â†’ `avg_kwh_per_user` is the better-fitting model.  
**Positive delta** â†’ `avg_kwh` is the better-fitting model.

In [31]:
# avg_kwh results: already computed above (comparison_dfs)
# avg_kwh_per_user results: generated by re-running this notebook with
#   TARGET_CANDIDATES = ['avg_kwh_per_user']  (see section 12 save cell for the output dir name)

PER_USER_DIR = BASE_PATH / 'forecasting_model_results_365_seasonality_v4_avg_kwh_per_user'
AVG_KWH_DIR  = BASE_PATH / 'forecasting_model_results_365_seasonality_v4_avg_kwh'

for group in GROUPS_TO_FORECAST:
    safe = str(group).replace('/', '_').replace(' ', '_')
    per_user_path = PER_USER_DIR / f'{safe}_metrics.csv'

    if not per_user_path.exists():
        print(f'[skip] avg_kwh_per_user results not found for {group}')
        print(f'       Re-run with TARGET_CANDIDATES = [\'avg_kwh_per_user\'] to generate them.')
        continue

    per_user = pd.read_csv(per_user_path).set_index('Model')[['MAPE(%)', 'sMAPE(%)']]
    avg_kwh  = comparison_dfs[group][['MAPE(%)', 'sMAPE(%)']]

    merged = per_user.join(avg_kwh, lsuffix='_per_user', rsuffix='_avg_kwh')
    merged['MAPE delta (pp)']  = merged['MAPE(%)_avg_kwh']  - merged['MAPE(%)_per_user']
    merged['sMAPE delta (pp)'] = merged['sMAPE(%)_avg_kwh'] - merged['sMAPE(%)_per_user']
    merged = merged.sort_values('MAPE(%)_per_user')

    print(f'\n{"="*100}')
    print(f'TARGET COMPARISON â€” {group}')
    print(f'  negative delta = avg_kwh_per_user is BETTER  |  positive delta = avg_kwh is BETTER')
    print(f'{"="*100}')
    display(merged.round(4))


TARGET COMPARISON â€” Vitoria-Gasteiz
  negative delta = avg_kwh_per_user is BETTER  |  positive delta = avg_kwh is BETTER


,MAPE(%)_per_user,sMAPE(%)_per_user,MAPE(%)_avg_kwh,sMAPE(%)_avg_kwh,MAPE delta (pp),sMAPE delta (pp)
Model,,,,,,
Random Forest Features,6.0984,6.0111,7.3459,7.6782,1.2475,1.6671
Seasonal Naive (365d),19.6481,17.3013,12.3026,11.7859,-7.3455,-5.5155
N-BEATS v2 (covariate-conditioned),26.1795,22.3345,5.3093,5.3047,-20.8701,-17.0299
Naive,31.3512,38.7527,31.2970,38.6875,-0.0542,-0.0653
Rolling Mean (7d),33.3658,41.7114,33.3135,41.6463,-0.0523,-0.0650
LSTM v2 (dual-stream),37.3164,41.9411,9.6673,9.9850,-27.6491,-31.9561
SARIMA,39.0677,28.1984,74.8513,132.3827,35.7836,104.1842
SARIMAX,100.0000,200.0000,52.9898,77.5525,-47.0102,-122.4475
Ridge Features,556.6349,162.0056,5.1256,5.1276,-551.5092,-156.8780



TARGET COMPARISON â€” Donostia/San Sebastian
  negative delta = avg_kwh_per_user is BETTER  |  positive delta = avg_kwh is BETTER


,MAPE(%)_per_user,sMAPE(%)_per_user,MAPE(%)_avg_kwh,sMAPE(%)_avg_kwh,MAPE delta (pp),sMAPE delta (pp)
Model,,,,,,
Random Forest Features,6.4844,4.8086,5.7807,6.0050,-0.7037,1.1963
SARIMA,14.9382,16.8774,12.9825,14.3795,-1.9558,-2.4979
Rolling Mean (7d),17.8792,20.5175,17.8335,20.4680,-0.0457,-0.0495
N-BEATS v2 (covariate-conditioned),18.6665,18.6165,3.3803,3.4003,-15.2862,-15.2162
Seasonal Naive (365d),21.5496,18.8517,10.6751,10.3150,-10.8746,-8.5367
Naive,22.4541,26.2656,22.4008,26.2052,-0.0533,-0.0604
LSTM v2 (dual-stream),49.6728,41.2427,6.0757,6.2644,-43.5971,-34.9783
Ridge Features,750.6536,165.2087,3.5305,3.6175,-747.1231,-161.5912
SARIMAX,3016.9842,176.5774,3.1526,3.0629,-3013.8316,-173.5144



TARGET COMPARISON â€” Pamplona/Iruna
  negative delta = avg_kwh_per_user is BETTER  |  positive delta = avg_kwh is BETTER


,MAPE(%)_per_user,sMAPE(%)_per_user,MAPE(%)_avg_kwh,sMAPE(%)_avg_kwh,MAPE delta (pp),sMAPE delta (pp)
Model,,,,,,
N-BEATS v2 (covariate-conditioned),5.1498,5.0821,4.4211,4.4406,-0.7287,-0.6415
Random Forest Features,7.8537,7.4675,3.1207,3.1852,-4.7330,-4.2823
LSTM v2 (dual-stream),7.9273,7.8977,6.8237,6.8763,-1.1035,-1.0214
SARIMA,14.5766,14.5743,14.9218,14.6424,0.3452,0.0681
Naive,16.5283,18.7987,16.4780,18.7399,-0.0502,-0.0588
Hybrid v2 SeasonalNaive(365d)+ResidualLSTM,16.9966,18.6944,22.0206,26.1121,5.0240,7.4177
Seasonal Naive (365d),18.6811,16.8485,9.2502,9.7151,-9.4309,-7.1334
Rolling Mean (7d),21.6061,25.1921,21.5065,25.0787,-0.0996,-0.1134
Ridge Features,26.2032,35.3598,3.8770,3.9074,-22.3261,-31.4524


## 14. Forecast Visualisations

Generates two publication-ready figures saved to `theses_text/images/`:

| File | Content |
|---|---|
| `forecast_comparison.png` | Actual vs. Seasonal Naive, LSTM v2, and N-BEATS v2 — one row per city |
| `forecast_residuals.png` | N-BEATS v2 residuals per city with ensemble-flagged anomaly days marked |

In [32]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from pathlib import Path

BASE_PATH = Path(r"C:\Users\GONCA\Desktop\Iscte\MCD\Theses")
PRED_DIR  = BASE_PATH / "forecasting_model_results_365_seasonality_v4_avg_kwh"
IMG_DIR   = BASE_PATH / "theses_text" / "images"

CITIES = [
    ("Vitoria-Gasteiz",         "Vitoria-Gasteiz"),
    ("Donostia/San Sebastian",  "Donostia_San_Sebastian"),
    ("Pamplona/Iruna",          "Pamplona_Iruna"),
]

ANOMALY_DAYS = {
    "Vitoria-Gasteiz":        ["2021-12-09", "2021-12-10", "2021-12-13"],
    "Donostia/San Sebastian": ["2021-12-09", "2022-01-07", "2022-01-24"],
    "Pamplona/Iruna":         ["2021-12-20", "2021-12-25"],
}

COL_NBEATS  = "N-BEATS v2 (covariate-conditioned)"
COL_LSTM    = "LSTM v2 (dual-stream)"
COL_SNAIVE  = "Seasonal Naive (365d)"

preds = {}
for label, slug in CITIES:
    df = pd.read_csv(PRED_DIR / f"{slug}_predictions.csv", parse_dates=["date"])
    preds[label] = df.set_index("date")

# ── Figure 1: Actual vs. key models ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)
fig.suptitle("Test-Set Forecast — Selected Models vs. Actual", fontsize=13, fontweight="bold")

for ax, (label, _) in zip(axes, CITIES):
    df = preds[label]
    ax.plot(df.index, df["actual"],   color="#1a1a1a", lw=1.8, label="Actual",             zorder=5)
    ax.plot(df.index, df[COL_SNAIVE], color="#aaaaaa", lw=1.2, ls="--", label="Seasonal Naive (365d)", zorder=2)
    ax.plot(df.index, df[COL_LSTM],   color="#3498db", lw=1.3, ls="--", label="LSTM v2",   zorder=3)
    ax.plot(df.index, df[COL_NBEATS], color="#e74c3c", lw=1.8, label="N-BEATS v2",         zorder=4)

    ax.set_title(label, fontsize=11, fontweight="bold", pad=5)
    ax.set_ylabel("avg kWh", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3, lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].legend(ncol=4, fontsize=8, loc="upper right", framealpha=0.9)
fig.tight_layout()
out1 = IMG_DIR / "forecast_comparison.png"
fig.savefig(out1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved {out1}")

# ── Figure 2: N-BEATS v2 residuals with anomaly days ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)
fig.suptitle("N-BEATS v2 Forecast Residuals — Anomalous Days Marked", fontsize=13, fontweight="bold")

for ax, (label, _) in zip(axes, CITIES):
    df   = preds[label]
    resid = df["actual"] - df[COL_NBEATS]
    colors = ["#e74c3c" if r > 0 else "#3498db" for r in resid]
    ax.bar(df.index, resid, color=colors, alpha=0.75, width=1.0)
    ax.axhline(0, color="black", lw=0.8)

    for day in ANOMALY_DAYS.get(label, []):
        d = pd.Timestamp(day)
        if d in resid.index:
            ax.axvline(d, color="#2c3e50", lw=1.4, ls="--", alpha=0.85)
            ax.annotate(day[5:], xy=(d, resid[d]),
                        xytext=(4, 4), textcoords="offset points",
                        fontsize=7, color="#2c3e50", fontweight="bold")

    ax.set_title(label, fontsize=11, fontweight="bold", pad=5)
    ax.set_ylabel("Residual (kWh)", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3, lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor="#e74c3c", alpha=0.75, label="Positive residual (under-forecast)"),
              Patch(facecolor="#3498db", alpha=0.75, label="Negative residual (over-forecast)")]
axes[0].legend(handles=legend_els, fontsize=8, loc="upper right", framealpha=0.9)
fig.tight_layout()
out2 = IMG_DIR / "forecast_residuals.png"
fig.savefig(out2, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved {out2}")

Saved C:\Users\GONCA\Desktop\Iscte\MCD\Theses\theses_text\images\forecast_comparison.png
Saved C:\Users\GONCA\Desktop\Iscte\MCD\Theses\theses_text\images\forecast_residuals.png
